In [3]:
from google.colab import files
uploaded = files.upload()

Saving capstone_repo_starter.zip to capstone_repo_starter (2).zip


In [4]:
import zipfile
with zipfile.ZipFile("capstone_repo_starter.zip", "r") as z:
    z.extractall(".")

In [5]:
!ls capstone_repo/data_pipeline

clean.py  db_setup.py  README.md	 scraper.py
data	  queries.py   requirements.txt


In [6]:
%cd capstone_repo/data_pipeline

/content/capstone_repo/data_pipeline


In [7]:
!pip install -r requirements.txt

In [8]:
!python scraper.py

Scraped 11 books from 'Travel' (running total: 11)
Scraped 32 books from 'Mystery' (running total: 43)
Scraped 26 books from 'Historical Fiction' (running total: 69)

Done. Wrote 69 rows across 3 categories -> /content/capstone_repo/data_pipeline/data/raw_books.csv


In [9]:
!python clean.py
!python db_setup.py
!python queries.py

Wrote 69 cleaned rows -> /content/capstone_repo/data_pipeline/data/clean_books.csv
Loaded 3 categories and 69 books -> /content/capstone_repo/data_pipeline/data/zepto_books.db

--- Q1: books under 500 INR (SELECT/WHERE) ---
SELECT title, price_inr FROM books WHERE price_inr < 500
['title', 'price_inr']
(0 row(s) total, showing up to 10)

--- Q2: 10 most expensive books (ORDER BY / LIMIT) ---
SELECT title, price_inr FROM books ORDER BY price_inr DESC LIMIT 10
['title', 'price_inr']
('Boar Island (Anna Pigeon #19)', 6275.14)
("The No. 1 Ladies' Detective Agency (No. 1 Ladies' Detective Agency #1)", 6087.35)
('A Year in Provence (Provence #1)', 6000.84)
('The Past Never Ends', 5960.75)
('The Last Painting of Sara de Vos', 5860.52)
('A Flight of Arrows (The Pathfinders #2)', 5858.42)
('Murder at the 42nd Street Library (Raymond Ambler #1)', 5734.98)
('The Last Mile (Amos Decker #2)', 5719.16)
("1st to Die (Women's Murder Club #1)", 5694.89)
('Tipping the Velvet', 5669.57)
(10 row(s) total,

In [10]:
def get_soup(url):
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")

In [11]:
def get_categories():
    """Return [(category_name, category_url), ...] parsed from the homepage sidebar."""
    soup = get_soup(BASE_URL)
    links = soup.select("div.side_categories ul li ul li a")
    return [(a.get_text(strip=True), BASE_URL + a["href"]) for a in links]

In [12]:
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://books.toscrape.com/"
resp = requests.get(BASE_URL, timeout=10)
soup = BeautifulSoup(resp.text, "html.parser")

links = soup.select("div.side_categories ul li ul li a")
categories = [(a.get_text(strip=True), BASE_URL + a["href"]) for a in links]

print(len(categories))
print(categories[:5])

50
[('Travel', 'https://books.toscrape.com/catalogue/category/books/travel_2/index.html'), ('Mystery', 'https://books.toscrape.com/catalogue/category/books/mystery_3/index.html'), ('Historical Fiction', 'https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html'), ('Sequential Art', 'https://books.toscrape.com/catalogue/category/books/sequential-art_5/index.html'), ('Classics', 'https://books.toscrape.com/catalogue/category/books/classics_6/index.html')]


In [13]:
"""
scraper.py
Scrapes book listings from books.toscrape.com across multiple categories.
Discovers categories dynamically from the site's own nav sidebar (rather than
hardcoding category URLs), then paginates through each category's listing
pages until we've collected a large enough, multi-category dataset.

Output: data_pipeline/data/raw_books.csv
Columns: title, price, star_rating, availability, category
"""
import csv
import os
import time
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://books.toscrape.com/"
OUTPUT_PATH = os.path.join("data", "raw_books.csv")
MIN_BOOKS = 60
MIN_CATEGORIES = 3


def get_soup(url):
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


def get_categories():
    soup = get_soup(BASE_URL)
    links = soup.select("div.side_categories ul li ul li a")
    return [(a.get_text(strip=True), BASE_URL + a["href"]) for a in links]


def scrape_category(category_name, category_url):
    books = []
    url = category_url
    while url:
        soup = get_soup(url)
        for pod in soup.select("article.product_pod"):
            title = pod.h3.a["title"].strip()
            price_text = pod.select_one("p.price_color").get_text(strip=True)
            rating_classes = pod.select_one("p.star-rating")["class"]
            rating_text = [c for c in rating_classes if c != "star-rating"][0]
            availability_text = pod.select_one("p.instock.availability").get_text(strip=True)
            books.append({
                "title": title,
                "price": price_text,
                "star_rating": rating_text,
                "availability": availability_text,
                "category": category_name,
            })
        next_link = soup.select_one("li.next a")
        if next_link:
            url = url.rsplit("/", 1)[0] + "/" + next_link["href"]
        else:
            url = None
        time.sleep(0.3)
    return books


def main():
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    categories = get_categories()

    all_books = []
    used_categories = 0
    for name, url in categories:
        books = scrape_category(name, url)
        if not books:
            continue
        all_books.extend(books)
        used_categories += 1
        print(f"Scraped {len(books)} books from '{name}' (running total: {len(all_books)})")
        if len(all_books) >= MIN_BOOKS and used_categories >= MIN_CATEGORIES:
            break

    if len(all_books) < MIN_BOOKS or used_categories < MIN_CATEGORIES:
        print("Below target after first pass -- scraping additional categories...")
        for name, url in categories[used_categories:]:
            books = scrape_category(name, url)
            if not books:
                continue
            all_books.extend(books)
            used_categories += 1
            print(f"Scraped {len(books)} books from '{name}' (running total: {len(all_books)})")
            if len(all_books) >= MIN_BOOKS:
                break

    with open(OUTPUT_PATH, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f, fieldnames=["title", "price", "star_rating", "availability", "category"]
        )
        writer.writeheader()
        writer.writerows(all_books)

    print(f"\nDone. Wrote {len(all_books)} rows across {used_categories} categories -> {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

Scraped 11 books from 'Travel' (running total: 11)
Scraped 32 books from 'Mystery' (running total: 43)
Scraped 26 books from 'Historical Fiction' (running total: 69)

Done. Wrote 69 rows across 3 categories -> data/raw_books.csv


In [14]:
import pandas as pd
df = pd.read_csv("data/raw_books.csv")
print(df.shape)
df.head()

(69, 5)


,title,price,star_rating,availability,category
0,It's Only the Himalayas,Â£45.17,Two,In stock,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,Â£49.43,Four,In stock,Travel
2,See America: A Celebration of Our National Par...,Â£48.87,Three,In stock,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,Â£36.94,Two,In stock,Travel
4,Under the Tuscan Sun,Â£37.33,Three,In stock,Travel


In [15]:
import pandas as pd
df = pd.read_csv("data/raw_books.csv")
print(df.shape)
df.head()

(69, 5)


,title,price,star_rating,availability,category
0,It's Only the Himalayas,Â£45.17,Two,In stock,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,Â£49.43,Four,In stock,Travel
2,See America: A Celebration of Our National Par...,Â£48.87,Three,In stock,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,Â£36.94,Two,In stock,Travel
4,Under the Tuscan Sun,Â£37.33,Three,In stock,Travel


In [16]:
import pandas as pd

RAW_PATH = "data/raw_books.csv"
CLEAN_PATH = "data/clean_books.csv"
GBP_TO_INR = 105.50

RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}


def clean_price(price_text):
    try:
        cleaned = str(price_text).replace("£", "").replace("Â", "").strip()
        return float(cleaned)
    except (ValueError, AttributeError, TypeError):
        return None


def clean_rating(rating_text):
    return RATING_MAP.get(str(rating_text).strip())


def clean_availability(avail_text):
    if avail_text is None:
        return None
    return "in stock" in str(avail_text).strip().lower()


df = pd.read_csv(RAW_PATH)

df["price_gbp"] = df["price"].apply(clean_price)
df["rating"] = df["star_rating"].apply(clean_rating)
df["in_stock"] = df["availability"].apply(clean_availability)

before = len(df)
df = df.dropna(subset=["price_gbp", "rating", "in_stock"])
dropped = before - len(df)
if dropped:
    print(f"Dropped {dropped} row(s) that failed to parse cleanly.")

df["price_inr"] = (df["price_gbp"] * GBP_TO_INR).round(2)
df["rating"] = df["rating"].astype(int)
df["in_stock"] = df["in_stock"].astype(bool)

df = df[["title", "price_gbp", "price_inr", "rating", "in_stock", "category"]]
df.to_csv(CLEAN_PATH, index=False)
print(f"Wrote {len(df)} cleaned rows -> {CLEAN_PATH}")

Wrote 69 cleaned rows -> data/clean_books.csv


In [17]:
import sqlite3
import pandas as pd

CLEAN_PATH = "data/clean_books.csv"
DB_PATH = "data/zepto_books.db"

SCHEMA = """
DROP TABLE IF EXISTS books;
DROP TABLE IF EXISTS categories;

CREATE TABLE categories (
    category_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE NOT NULL
);

CREATE TABLE books (
    book_id     INTEGER PRIMARY KEY AUTOINCREMENT,
    title       TEXT NOT NULL,
    price_gbp   REAL NOT NULL,
    price_inr   REAL NOT NULL,
    rating      INTEGER NOT NULL,
    in_stock    INTEGER NOT NULL,
    category_id INTEGER NOT NULL REFERENCES categories(category_id)
);
"""

df = pd.read_csv(CLEAN_PATH)

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.executescript(SCHEMA)

categories = sorted(df["category"].unique())
cur.executemany("INSERT INTO categories (category_name) VALUES (?)", [(c,) for c in categories])
conn.commit()

cat_id_map = dict(cur.execute("SELECT category_name, category_id FROM categories").fetchall())

rows = [
    (r.title, r.price_gbp, r.price_inr, int(r.rating), int(bool(r.in_stock)), cat_id_map[r.category])
    for r in df.itertuples(index=False)
]
cur.executemany(
    """INSERT INTO books (title, price_gbp, price_inr, rating, in_stock, category_id)
       VALUES (?, ?, ?, ?, ?, ?)""",
    rows,
)
conn.commit()
conn.close()

print(f"Loaded {len(categories)} categories and {len(rows)} books -> {DB_PATH}")

Loaded 3 categories and 69 books -> data/zepto_books.db


In [18]:
import sqlite3
import pandas as pd

DB_PATH = "data/zepto_books.db"

def run(cur, label, sql):
    print(f"\n--- {label} ---")
    print(sql.strip())
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description]
    print(cols)
    for row in rows[:10]:
        print(row)
    print(f"({len(rows)} row(s) total, showing up to 10)")

JOIN_SQL = """
    SELECT c.category_name, b.title, b.rating
    FROM books b
    JOIN categories c ON b.category_id = c.category_id
    WHERE b.rating >= 4
    ORDER BY c.category_name, b.rating DESC
"""

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

run(cur, "Q1: books under 500 INR", "SELECT title, price_inr FROM books WHERE price_inr < 500")
run(cur, "Q2: 10 most expensive books", "SELECT title, price_inr FROM books ORDER BY price_inr DESC LIMIT 10")
run(cur, "Q3: distinct category names", "SELECT DISTINCT category_name FROM categories")
run(cur, "Q4: 4-5 star books, 300-1500 INR", "SELECT title, rating, price_inr FROM books WHERE rating IN (4,5) AND price_inr BETWEEN 300 AND 1500")
run(cur, "Q5: books rated >=4 per category (JOIN)", JOIN_SQL)

df_q1 = pd.read_sql("SELECT title, price_inr FROM books WHERE price_inr < 500", conn)
df_q5_sql = pd.read_sql(JOIN_SQL, conn)
print("\n--- pd.read_sql Q1 head ---"); print(df_q1.head())
print("\n--- pd.read_sql Q5 head ---"); print(df_q5_sql.head())

books_df = pd.read_sql("SELECT * FROM books", conn)
categories_df = pd.read_sql("SELECT * FROM categories", conn)
merged = books_df.merge(categories_df, on="category_id")
merged = merged[merged["rating"] >= 4][["category_name", "title", "rating"]]
merged = merged.sort_values(["category_name", "rating"], ascending=[True, False]).reset_index(drop=True)
df_q5_sorted = df_q5_sql.sort_values(["category_name", "rating"], ascending=[True, False]).reset_index(drop=True)

print("\n--- pd.merge equivalent head ---"); print(merged.head())
print(f"\nSQL join result and pandas merge result match: {merged.equals(df_q5_sorted)}")

conn.close()


--- Q1: books under 500 INR ---
SELECT title, price_inr FROM books WHERE price_inr < 500
['title', 'price_inr']
(0 row(s) total, showing up to 10)

--- Q2: 10 most expensive books ---
SELECT title, price_inr FROM books ORDER BY price_inr DESC LIMIT 10
['title', 'price_inr']
('Boar Island (Anna Pigeon #19)', 6275.14)
("The No. 1 Ladies' Detective Agency (No. 1 Ladies' Detective Agency #1)", 6087.35)
('A Year in Provence (Provence #1)', 6000.84)
('The Past Never Ends', 5960.75)
('The Last Painting of Sara de Vos', 5860.52)
('A Flight of Arrows (The Pathfinders #2)', 5858.42)
('Murder at the 42nd Street Library (Raymond Ambler #1)', 5734.98)
('The Last Mile (Amos Decker #2)', 5719.16)
("1st to Die (Women's Murder Club #1)", 5694.89)
('Tipping the Velvet', 5669.57)
(10 row(s) total, showing up to 10)

--- Q3: distinct category names ---
SELECT DISTINCT category_name FROM categories
['category_name']
('Historical Fiction',)
('Mystery',)
('Travel',)
(3 row(s) total, showing up to 10)

--- Q

In [19]:
import os
print(os.getcwd())
print(os.listdir("."))

/content/capstone_repo/data_pipeline
['data', 'scraper.py', 'db_setup.py', 'queries.py', 'requirements.txt', 'clean.py', 'README.md']


In [20]:
import os
for root, dirs, files in os.walk("/content"):
    if "raw_books.csv" in files or "clean_books.csv" in files or "zepto_books.db" in files:
        print(root, files)

/content/capstone_repo/data_pipeline/data ['zepto_books.db', 'raw_books.csv', 'clean_books.csv']
